# Residual CO2 emissions in IPCC AR6 mitigation scenarios

**Question:** in scenarios that limit warming to 1.5°C or 2°C, how much fossil and industrial CO2
is still emitted around mid-century, and how much carbon dioxide removal balances it?

**Data:** IPCC AR6 Scenarios Database (World), hosted by IIASA, read with
[pyam](https://pyam-iamc.readthedocs.io/).

**Scenario categories:** C1 = limit warming to 1.5°C with no or limited overshoot;
C3 = likely limit warming to 2°C.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyam

# Only needed on machines where antivirus software inspects secure connections
try:
    import truststore
    truststore.inject_into_ssl()
except ImportError:
    pass

## 1. Scenario categories

Each AR6 scenario has a climate category in the database's metadata. We keep C1 and C3.

In [ ]:
CATS = {"C1": "1.5°C (no or limited overshoot)", "C3": "Likely below 2°C"}

conn = pyam.iiasa.Connection("ar6-public")
meta = conn.meta().reset_index()
meta = meta.loc[meta["Category"].isin(CATS), ["model", "scenario", "Category"]]
meta["Category"].value_counts()

## 2. Emissions and removals data

Five global variables, downloaded once and saved to `data/` (about 2 MB):

| Variable | Meaning |
|---|---|
| `Emissions|CO2|Energy and Industrial Processes` | Net fossil and industrial CO2, after BECCS and DACCS |
| `Carbon Sequestration|CCS|Biomass` | CO2 captured and stored from bioenergy (BECCS) |
| `Carbon Sequestration|Direct Air Capture` | CO2 captured from the air and stored (DACCS) |
| `Emissions|CO2`, `Emissions|CO2|AFOLU` | Total and land-use CO2, kept for reference |

In [ ]:
DATA = Path("data/ar6_world_co2.csv")
VARS = [
    "Emissions|CO2",
    "Emissions|CO2|Energy and Industrial Processes",
    "Emissions|CO2|AFOLU",
    "Carbon Sequestration|CCS|Biomass",
    "Carbon Sequestration|Direct Air Capture",
]

if not DATA.exists():
    DATA.parent.mkdir(exist_ok=True)
    conn.query(variable=VARS, region="World").to_csv(DATA)

df = pyam.IamDataFrame(DATA)
df

## 3. From net to gross emissions

Net fossil and industrial CO2 already subtracts the CO2 stored by BECCS and DACCS.
Adding those removals back gives the **gross** emissions that are still released:

`gross = net energy & industry CO2 + BECCS + DACCS`

Scenarios that do not report BECCS or DACCS are counted as zero for that removal.
Values are converted from Mt to Gt CO2 per year.

In [ ]:
EIP = "Emissions|CO2|Energy and Industrial Processes"
BECCS = "Carbon Sequestration|CCS|Biomass"
DACCS = "Carbon Sequestration|Direct Air Capture"

wide = (df.timeseries()
        .reset_index()
        .drop(columns=["region", "unit"])
        .melt(id_vars=["model", "scenario", "variable"], var_name="year")
        .pivot_table(index=["model", "scenario", "year"], columns="variable", values="value")
        .reset_index())
wide["year"] = wide["year"].astype(int)
wide = wide.merge(meta, on=["model", "scenario"]).dropna(subset=[EIP])

wide["removals"] = wide[[BECCS, DACCS]].fillna(0).sum(axis=1)
wide["gross"] = wide[EIP] + wide["removals"]
wide = wide.rename(columns={EIP: "net"})
wide[["net", "gross", "removals"]] = wide[["net", "gross", "removals"]] / 1000  # Gt CO2/yr
wide.head()

## 4. Mid-century: the year 2050

In [ ]:
y2050 = wide[wide["year"] == 2050]
(y2050.groupby("Category")[["gross", "removals", "net"]]
      .median()
      .round(1)
      .assign(n_scenarios=y2050.groupby("Category").size()))

## 5. Figure 1: gross and net emissions over time in 1.5°C scenarios

In [ ]:
Path("figures").mkdir(exist_ok=True)
c1 = wide[(wide["Category"] == "C1") & wide["year"].between(2020, 2100) & (wide["year"] % 5 == 0)]

fig, ax = plt.subplots(figsize=(8, 4.5))
for col, label, color in [("gross", "Gross fossil & industry CO2 (before removals)", "#c0392b"),
                          ("net", "Net fossil & industry CO2 (after BECCS & DACCS)", "#2c3e50")]:
    q = c1.groupby("year")[col].quantile([0.25, 0.5, 0.75]).unstack()
    ax.fill_between(q.index, q[0.25], q[0.75], color=color, alpha=0.2)
    ax.plot(q.index, q[0.5], color=color, lw=2, label=label)
ax.axhline(0, color="grey", lw=0.8)
ax.set_ylabel("Gt CO2 per year")
ax.set_title("1.5°C scenarios (IPCC AR6, C1): median and interquartile range")
ax.legend(frameon=False, loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig("figures/fig1_gross_vs_net_C1.png", dpi=150)

## 6. Figure 2: residual gross emissions in 2050, 1.5°C vs 2°C

In [ ]:
rng = np.random.default_rng(1)
data = [y2050.loc[y2050["Category"] == c, "gross"] for c in CATS]

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.boxplot(data, widths=0.5, showfliers=False, medianprops={"color": "#2c3e50", "lw": 2})
for i, d in enumerate(data, start=1):
    ax.scatter(i + rng.uniform(-0.15, 0.15, len(d)), d, s=10, alpha=0.4, color="#c0392b")
ax.set_xticks([1, 2], [f"{c}\n{name}\n(n={len(d)})" for (c, name), d in zip(CATS.items(), data)])
ax.set_ylabel("Gt CO2 per year in 2050")
ax.set_title("Residual gross fossil & industry CO2 in 2050")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig("figures/fig2_residual_2050.png", dpi=150)

## 7. What this shows, and its limits

- Net fossil and industrial CO2 falls steeply in 1.5°C scenarios, but **gross** emissions do not reach
  zero: a residual remains around mid-century and beyond, balanced by BECCS and DACCS.
- Residual gross emissions in 2050 are clearly lower in 1.5°C (C1) than in 2°C (C3) scenarios,
  but the spread across models is wide.

**Limits:** gross emissions are approximated from three reported variables; scenarios that do not
report BECCS or DACCS are treated as having none; land-use CO2 and removals (AFOLU) are not included
in the gross figure; later years have fewer reporting scenarios.